# Cross-contour transport

Volume transport across a closed bathymetry contour, for example
onto the Antarctic continental shelf across the 1000 m isobath.

The upstream notebook carries a standing warning that its contour
and mask construction can produce contours that are **not closed**
([#669](https://github.com/COSIMA/cosima-recipes/issues/669)),
together with non-monotonic contour indexing
([#383](https://github.com/COSIMA/cosima-recipes/issues/383)).
That is the part this translation changes rather than copies:

- `contour_cells` inserts an intermediate cell at every diagonal
  step, so the cell sequence is adjacently connected by
  construction;
- `cross_contour_masks` then assigns exactly one signed face
  crossing per step.

Both properties are asserted in
`validation/test_cosima_common.py`, including that a closed
contour has balanced crossings in each direction, which is the
check that catches an unclosed contour.

Positive transport is outward, away from the shallow side.

---

Translated from the COSIMA recipe
[`Cross-contour_transport.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Cross-contour_transport.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
from esmvalcore.preprocessor import (climate_statistics,
                                     extract_region)

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Why nothing is regridded here, but `extract_region` stays

`umo` and `vmo` are **extensive**: kg s$^{-1}$ through one cell
face, not a flux per unit area. Interpolating them gives each new
cell a plausible value without rescaling it for the new cell
size, so the total changes with the number of cells. The
0.5x0.5 regrid this notebook used to do inflated the transport
by **3.8x** — silently, because a cross-shelf transport of the
wrong magnitude still looks like a cross-shelf transport.

`extract_region` is a different matter and is kept. It *selects*
cells rather than interpolating between them, so every surviving
cell keeps exactly its own transport. It also keeps the contour
search on the Antarctic margin rather than letting the longest
1000 m contour anywhere on Earth win.

Nothing here needs a regular grid anyway: the contour is traced
in **index space** on `deptho`, and the face masks are applied to
`umo`/`vmo` on that same grid. Working natively is both correct
and simpler.

One consequence for plotting: on a native tripolar grid
`latitude` and `longitude` are 2-D, so the map below uses
`lat_2d`/`lon_2d`. A representative 1-D latitude would draw the
contour in the wrong place.

## Load the data

In [ ]:
common = dict(project='CMIP6', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              dataset='ACCESS-CM2')

umo = Dataset(short_name='umo', mip='Omon',
              timerange='2000/2009', **common)
vmo = umo.copy(short_name='vmo')
deptho = Dataset(short_name='deptho', mip='Ofx', **common)

def southern(cube):
    """Select the Antarctic margin. No regrid: see above."""
    return extract_region(cube, start_longitude=0.,
                          end_longitude=360.,
                          start_latitude=-82., end_latitude=-59.)

umo_cube = southern(climate_statistics(umo.load(),
                                       operator='mean',
                                       period='full'))
vmo_cube = southern(climate_statistics(vmo.load(),
                                       operator='mean',
                                       period='full'))
deptho_cube = southern(deptho.load())

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import logging

LOGGER = logging.getLogger("cosima_cmip")


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


RHO0 = 1035.0  # kg m-3, Boussinesq reference density used by MOM5/MOM6


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def lon_2d(cube):
    """2D longitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("longitude")
    if coord.ndim == 2:
        return coord.points
    lat = cube.coord("latitude").points
    return np.broadcast_to(coord.points[None, :], (lat.size, coord.shape[0]))


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def check_transport_grid(cube, name="transport"):
    """Warn if an extensive transport looks like it has been regridded.

    ``umo`` and ``vmo`` are *extensive*: kg s-1 through one cell face,
    not a flux per unit area.  Interpolating them and then summing
    scales the total with the number of cells, so a 1x1 to 0.5x0.5
    regrid inflates a transport by roughly four.  Nothing in the
    interpolation complains -- the numbers stay plausible -- so this
    checks the one thing that is visible after the fact: whether the
    cube still sits on the model's own (curvilinear) grid.

    A regular lat/lon grid is not proof of regridding, since a few
    models publish ocean data on one, hence a warning rather than an
    error.
    """
    if cube.coord("latitude").ndim == 1:
        LOGGER.warning(
            "%s is on a regular lat/lon grid. If it was regridded, its "
            "total is wrong: %s is kg s-1 per cell face, and interpolation "
            "does not rescale it for the new cell size. Compute transports "
            "on the native grid, or convert to a flux per unit area before "
            "regridding and multiply by the new face area afterwards.",
            name, name)
        return False
    return True


def bathymetry_on_grid(deptho_cube, reference_cube):
    """Bathymetry on the same horizontal grid as ``reference_cube``.

    ``deptho`` lives on the tracer grid while ``umo``/``vmo`` live on
    cell faces.  On a native (unregridded) grid those are genuinely
    different arrays: ``extract_region`` selects by latitude *value*,
    and because the staggered latitudes sit half a cell away it can
    return one row fewer for the transports than for ``deptho``.  A
    contour traced on the tracer grid then cannot index the transports
    at all -- shapes ``(50, 45, 360)`` against ``(50, 44, 360)``, which
    is exactly how this failed on ACCESS-CM2.

    Where the shapes already agree, ``deptho`` is used as it is.  Where
    they do not, the bathymetry comes from the reference field's own
    deepest wet cell, which is on the right grid by construction.  That
    is the same substitution ``Along_Isobath_Average.ipynb`` makes for
    its own reasons, and it costs the continuous depth: the contour is
    then quantised to the model's vertical levels.

    Returns ``(bathymetry, source)``.
    """
    reference_shape = reference_cube.shape[-2:]
    if deptho_cube is not None:
        depth = masked_data(deptho_cube)
        if depth.shape == reference_shape:
            return depth, "deptho"
        LOGGER.warning(
            "deptho is %s but the field it must index is %s -- these are "
            "staggered grids that extract_region has trimmed differently. "
            "Falling back to the bottom depth implied by the field itself, "
            "which is quantised to the model levels. To use deptho, put "
            "both on a common grid -- but never regrid a transport.",
            depth.shape, reference_shape)

    depth_name = depth_coord_name(reference_cube)
    if depth_name is None:
        raise ValueError(
            "no deptho on the right grid and no depth axis to derive one")
    levels = reference_cube.coord(depth_name).points
    data = masked_data(reference_cube)
    while data.ndim > 3:
        data = data[0]
    marker = np.ma.masked_array(
        np.broadcast_to(levels[:, None, None], data.shape).astype(float),
        mask=np.ma.getmaskarray(data))
    valid = ~np.ma.getmaskarray(marker)
    any_valid = valid.any(axis=0)
    deepest = valid.shape[0] - 1 - np.argmax(valid[::-1], axis=0)
    bathymetry = np.take_along_axis(
        np.ma.filled(marker, np.nan), deepest[None, ...], axis=0)[0]
    return (np.ma.masked_array(bathymetry, mask=~any_valid),
            "deepest wet level of the transport field")


def extract_contour(field, level, longest_only=True):
    """Contour of ``field`` at ``level``, in index space.

    ``field`` is a 2D array, typically ``deptho``.  Returns ``(x, y)``
    fractional index coordinates.  As in ``Create_contour.ipynb`` only
    the longest path is kept by default, which discards contours around
    isolated seamounts and depressions.
    """
    import matplotlib

    matplotlib.use("Agg", force=False)
    import matplotlib.pyplot as plt

    data = np.ma.masked_invalid(np.ma.asarray(field, dtype=float))
    figure = plt.figure()
    try:
        axes = figure.add_subplot(111)
        contour_set = axes.contour(np.ma.filled(data, 0.0), levels=[level])
        paths = contour_set.get_paths()
        segments = []
        for path in paths:
            vertices = path.vertices
            if vertices.shape[0] < 2:
                continue
            # A single Path may hold several disjoint polylines; split
            # them where the step between consecutive vertices jumps.
            steps = np.abs(np.diff(vertices[:, 0]).astype(int))
            breaks = np.where(steps > 1)[0]
            start = 0
            for end in list(breaks) + [vertices.shape[0] - 1]:
                if end + 1 - start >= 2:
                    segments.append(vertices[start:end + 1])
                start = end + 1
    finally:
        plt.close(figure)

    if not segments:
        return np.array([]), np.array([])
    if longest_only:
        chosen = max(segments, key=lambda seg: seg.shape[0])
        return chosen[:, 0], chosen[:, 1]
    stacked = np.concatenate(segments, axis=0)
    return stacked[:, 0], stacked[:, 1]


def contour_cells(x_contour, y_contour, shape):
    """Snap a contour to unique, adjacently-connected grid cells.

    Returns ``(rows, cols)`` integer index arrays ordered along the
    contour.  Diagonal steps are filled in with an intermediate cell,
    which ``Cross-contour_transport.ipynb`` requires for the transport
    masks to be exact.
    """
    rows = np.clip(np.rint(y_contour).astype(int), 0, shape[0] - 1)
    cols = np.clip(np.rint(x_contour).astype(int), 0, shape[1] - 1)

    out_rows, out_cols = [], []
    for row, col in zip(rows, cols):
        if out_rows and out_rows[-1] == row and out_cols[-1] == col:
            continue
        if out_rows:
            drow = row - out_rows[-1]
            dcol = col - out_cols[-1]
            if abs(drow) == 1 and abs(dcol) == 1:
                # Insert the cell that makes the step adjacent rather
                # than diagonal; step in x first, as the notebook does.
                out_rows.append(out_rows[-1])
                out_cols.append(col)
        out_rows.append(row)
        out_cols.append(col)
    return np.array(out_rows), np.array(out_cols)


def cross_contour_masks(rows, cols, shape):
    """Masks selecting the faces crossed by a contour, with sign.

    Returns ``(mask_u, mask_v)``, each of shape ``shape``, holding +1 or
    -1 on the cells whose eastward (``mask_u``) or northward
    (``mask_v``) face is crossed by the contour, and 0 elsewhere.  The
    sign is set so that the summed transport is positive *outward*, away
    from the shallow side of the contour.

    Multiplying the mask by ``umo``/``vmo`` and summing gives the
    cross-contour transport of ``Cross-contour_transport.ipynb``.
    """
    rows = np.asarray(rows, dtype=int)
    cols = np.asarray(cols, dtype=int)
    if (rows.size != cols.size or not rows.size
            or np.any(rows < 0) or np.any(rows >= shape[0])
            or np.any(cols < 0) or np.any(cols >= shape[1])):
        raise ValueError(
            "contour indices do not fit the transport grid "
            f"{tuple(shape)}. deptho and umo/vmo must cover the same "
            "horizontal region; do not clip or interpolate mass transport.")

    mask_u = np.zeros(shape)
    mask_v = np.zeros(shape)
    for index in range(len(rows) - 1):
        row, col = rows[index], cols[index]
        drow = rows[index + 1] - row
        dcol = cols[index + 1] - col
        # Walking the contour with the shallow side on the left, a step
        # in +x crosses the northward face and a step in +y crosses the
        # eastward face.
        if dcol == 1:
            mask_v[row, col] += 1
        elif dcol == -1:
            mask_v[row, col] -= 1
        elif drow == 1:
            mask_u[row, col] -= 1
        elif drow == -1:
            mask_u[row, col] += 1
    return mask_u, mask_v


def common_face_shape(*shapes):
    """Overlap on which native staggered transport faces can be indexed.

    MOM's ``umo`` and ``vmo`` live on different faces.  After a geographic
    ``extract_region`` their latitude coordinates can select one fewer edge
    row from one component: ACCESS-CM2, for example, yields ``(45, 360)``
    and ``(44, 360)``.  That is a normal C-grid edge, not a reason to
    interpolate either extensive transport.

    The common index region is the shorter latitude dimension.  It is only
    valid for this one-edge-row stagger; different zonal dimensions or a
    larger mismatch are evidence that the fields do not cover the same
    region, and must fail rather than silently discard transports.
    """
    horizontal = [tuple(shape[-2:]) for shape in shapes]
    if not horizontal or any(len(shape) != 2 or min(shape) <= 0
                             for shape in horizontal):
        raise ValueError("transport fields need non-empty horizontal grids")
    rows = [shape[0] for shape in horizontal]
    columns = [shape[1] for shape in horizontal]
    if len(set(columns)) != 1:
        raise ValueError(
            f"transport grids have different zonal dimensions: {horizontal}")
    if max(rows) - min(rows) > 1:
        raise ValueError(
            "transport grids differ by more than one latitude edge row: "
            f"{horizontal}; check that extract_region selected the same "
            "native domain")
    return min(rows), columns[0]


def cross_contour_faces(rows, cols, shape):
    """Ordered signed faces crossed by a contour.

    Returns ``(rows, cols, components, signs)`` in contour order, where
    ``components`` is ``"u"`` or ``"v"``.  Unlike the 2-D masks this keeps
    the path ordering needed for the original recipe's cumulative
    along-contour transport plot.
    """
    # Reuse the mask constructor's strict grid validation.  A silent clip
    # would turn a mass-transport integral into a different contour.
    cross_contour_masks(rows, cols, shape)
    face_rows, face_cols, components, signs = [], [], [], []
    for index in range(len(rows) - 1):
        row, col = rows[index], cols[index]
        drow = rows[index + 1] - row
        dcol = cols[index + 1] - col
        if dcol == 1:
            component, sign = "v", 1.0
        elif dcol == -1:
            component, sign = "v", -1.0
        elif drow == 1:
            component, sign = "u", -1.0
        elif drow == -1:
            component, sign = "u", 1.0
        else:
            continue
        face_rows.append(row)
        face_cols.append(col)
        components.append(component)
        signs.append(sign)
    return (np.asarray(face_rows, dtype=int), np.asarray(face_cols, dtype=int),
            np.asarray(components), np.asarray(signs, dtype=float))


def describe_contour(rows, cols, mask_u, mask_v, shape):
    """Classify a contour and apply the right closure check.

    There are two kinds of contour here and they close differently.

    A *loop* -- around a seamount or a basin -- returns to its start,
    so every face it crosses one way it crosses back and both mask
    sums are zero.

    A *circumpolar* contour, which is what the 1000 m isobath around
    Antarctica is, does not close in index space at all: it wraps
    through the periodic zonal boundary. Walking it eastward crosses
    one northward face per column, so ``mask_v`` sums to roughly the
    number of columns rather than to zero. That is correct, and a
    check demanding zero would report the Antarctic case as broken --
    which is exactly what the first Gadi run did, printing
    "v-face: +359 (0 means closed)".

    Returns ``(kind, message)``.
    """
    n_columns = shape[-1]
    spans_globe = (cols.max() - cols.min()) >= 0.9 * (n_columns - 1)
    v_sum, u_sum = float(mask_v.sum()), float(mask_u.sum())

    if spans_globe and abs(abs(v_sum) - n_columns) <= 0.1 * n_columns:
        return "circumpolar", (
            f"circumpolar contour: {rows.size} cells spanning the full "
            f"zonal range, {v_sum:+.0f} net northward-face crossings for "
            f"{n_columns} columns -- one per column, as it should be")
    if not spans_globe and abs(v_sum) < 2 and abs(u_sum) < 2:
        return "loop", (
            f"closed loop: {rows.size} cells, face crossings balance "
            f"({u_sum:+.0f} zonal, {v_sum:+.0f} meridional)")
    return "open", (
        f"contour does not close: {rows.size} cells, {u_sum:+.0f} zonal "
        f"and {v_sum:+.0f} meridional net crossings over {n_columns} "
        f"columns. The transport across it is not a well-defined "
        f"exchange; check contour_depth and the region.")


def contour_indices_to_coordinates(rows, cols, lat, lon):
    """Latitude/longitude of integer contour cells."""
    lat = np.asarray(lat)
    lon = np.asarray(lon)
    if lat.ndim == 2:
        return lat[rows, cols], lon[rows, cols]
    return lat[rows], lon[cols]

## Build the contour and its transport masks

In [ ]:
CONTOUR_DEPTH = 1000.0

# `deptho` is on tracer points, while native C-grid transports
# can retain one fewer edge row after geographic extraction.
# Use deptho when it fits; otherwise derive bottom depth on the
# vmo grid. Never interpolate the extensive transports.
shape = common_face_shape(umo_cube.shape[-2:],
                          vmo_cube.shape[-2:])
contour_grid_cube = (umo_cube if umo_cube.shape[-2:] == shape
                     else vmo_cube)
depth, depth_source = bathymetry_on_grid(deptho_cube,
                                          contour_grid_cube)
depth_cube = (deptho_cube if depth_source == 'deptho'
              else contour_grid_cube)
x_contour, y_contour = extract_contour(
    np.ma.filled(depth, 0.0), CONTOUR_DEPTH)
rows, cols = contour_cells(x_contour, y_contour, depth.shape)
print(f'contour has {rows.size} cells')
contour_lat, contour_lon = contour_indices_to_coordinates(
    rows, cols, lat_2d(depth_cube), lon_2d(depth_cube))

# Sanity: adjacent, never diagonal.
steps = np.abs(np.diff(rows)) + np.abs(np.diff(cols))
assert np.all(steps <= 1), 'contour has a diagonal step'

## Transport

In [ ]:
# Warns if the transports have ended up on a regular grid,
# which would mean their totals are no longer the model's.
check_transport_grid(umo_cube, 'umo')
check_transport_grid(vmo_cube, 'vmo')

u_data = masked_data(umo_cube)
v_data = masked_data(vmo_cube)

# The native C-grid components can differ by one unused edge
# row after geographic extraction. Build masks only on their
# shared interior index region; never interpolate the extensive
# transports or broadcast one component's mask onto the other.
shape = common_face_shape(u_data.shape[-2:], v_data.shape[-2:])
mask_u, mask_v = cross_contour_masks(rows, cols, shape)
kind, message = describe_contour(rows, cols, mask_u, mask_v,
                                 shape)
if kind == 'open':
    raise ValueError(message)
print(message)

face_rows, face_cols, components, signs = cross_contour_faces(
    rows, cols, shape)
transport_faces = np.ma.masked_all(
    (u_data.shape[0], len(face_rows)))
for face, (row, col, component, sign) in enumerate(
        zip(face_rows, face_cols, components, signs)):
    values = u_data[:, row, col] if component == 'u' else                     v_data[:, row, col]
    transport_faces[:, face] = sign * values / RHO0 / 1.0e6

depth_levels = umo_cube.coord(
    depth_coord_name(umo_cube)).points
DEEPER_THAN = 800.0
deep = depth_levels >= DEEPER_THAN
cumulative = np.ma.cumsum(np.ma.sum(transport_faces[deep],
                                    axis=0))

# Great-circle distance along the ordered contour, used only for
# plotting as in the original notebook.
dlat = np.deg2rad(np.diff(contour_lat)[:len(face_rows)])
dlon = np.deg2rad(np.diff(contour_lon)[:len(face_rows)])
lat0 = np.deg2rad(contour_lat[:-1][:len(face_rows)])
lat1 = np.deg2rad(contour_lat[1:][:len(face_rows)])
a = np.sin(dlat / 2)**2 + np.cos(lat0) * np.cos(lat1) *                 np.sin(dlon / 2)**2
distance = np.r_[0., np.cumsum(
    2 * EARTH_RADIUS * np.arcsin(np.sqrt(np.clip(a, 0, 1))))]
distance = distance[:len(cumulative)] / 1.e3
print(f'net transport below {DEEPER_THAN:.0f} m: '
      f'{cumulative[-1]:+.2f} Sv')

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Native tripolar coordinate centres are non-monotonic at the
# fold. ``pcolormesh`` would infer spurious cell corners there,
# so draw the actual centres instead of inventing edges.
map_lat, map_lon = lat_2d(depth_cube), lon_2d(depth_cube)
valid = ~np.ma.getmaskarray(depth)
axes[0].scatter(map_lon[valid], map_lat[valid], c=depth[valid],
                s=4, marker='s', linewidths=0, cmap='Blues',
                rasterized=True)
contour_lat, contour_lon = contour_indices_to_coordinates(
    rows, cols, lat_2d(depth_cube), lon_2d(depth_cube))
axes[0].plot(contour_lon, contour_lat, color='red', linewidth=1.0)
axes[0].set_title(f'{CONTOUR_DEPTH:.0f} m contour')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')

axes[1].plot(distance, cumulative)
axes[1].axhline(0, color='k', linewidth=0.6)
axes[1].set_xlabel('Distance along contour (km)')
axes[1].set_ylabel('Cumulative transport (Sv), positive outward')
axes[1].set_title(f'Cumulative transport below '
                  f'{DEEPER_THAN:.0f} m')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()